# SPEC2RTL & PROMPT2SILICON: HUẤN LUYỆN SFT LORA TỪ DỮ LIỆU CHƯNG CẤT GEMINI TEACHER
### Đề tài: Hệ thống sinh và kiểm chứng mã Verilog từ đặc tả ngôn ngữ tự nhiên
**Nhóm 2:** Huấn luyện AI Cục bộ (Fine-tuning SFT LoRA trên dữ liệu đã qua AST Linter + ModelSim Truth Oracle)

- **Chế độ dữ liệu mặc định (`USE_PURE_TEACHER_DISTILL = True`):** **1,637 mẫu** 100% thuần Gemini Teacher CoT + Verilog IEEE 1364-2001 (từ `dataset_teacher_distilled.json`).
- **Chế độ tập gộp (`USE_PURE_TEACHER_DISTILL = False`):** **2,724 mẫu** hợp nhất toàn kho sạch (`dataset_clean_verified.json`).
- **Mô hình mặc định:** `Qwen2.5-Coder-3B-Instruct` (4-bit QLoRA trên T4 GPU 15GB VRAM miễn phí).

## Bước 1: Kiểm tra cấu hình GPU Google Colab
*Lưu ý: Vào menu **Runtime -> Change runtime type -> T4 GPU** (miễn phí).*

In [ ]:
!nvidia-smi
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Bước 2: Cài đặt thư viện Unsloth & TRL
*Unsloth giúp tăng tốc huấn luyện ~2x và tiết kiệm ~70% VRAM trên Colab T4.*

In [ ]:
%%capture
!pip install -q unsloth matplotlib

## Bước 3: Nạp mô hình nền `Qwen2.5-Coder` (QLoRA 4-bit)
Có thể đổi `MODEL_SIZE` sang `"0.5B"`, `"1.5B"`, hoặc `"3B"` tùy cấu hình máy chạy local sau này.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_SIZE = "3B"
MODEL_MAP = {
    "0.5B": "unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit",
    "1.5B": "unsloth/Qwen2.5-Coder-1.5B-Instruct-bnb-4bit",
    "3B":   "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit",
}
model_name = MODEL_MAP[MODEL_SIZE]

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print(f">>> Nạp mô hình nền {model_name} thành công!")

## Bước 4: Cấu hình tham số LoRA (Low-Rank Adaptation)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print(f">>> Đã gắn LoRA Adapters vào {model_name}!")

## Bước 5: Tải và định dạng tập dữ liệu chưng cất từ GitHub (`LamGianggg/TKLLS`)

In [ ]:
import json, urllib.request, os
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

SYSTEM_PROMPT = (
    "Bạn là chuyên gia thiết kế vi mạch RTL và Verilog synthesizable chuẩn công nghiệp IEEE 1364-2001. "
    "Luôn sinh mã chuẩn sạch, đầy đủ cổng, không tạo chốt ẩn (no latches)."
)

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        user_content = f"{instruction}\n\n[Đặc tả thiết kế]:\n{input_text}" if input_text else instruction
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": output}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
        texts.append(text)
    return {"text": texts}

# True = 1,637 mẫu 100% thuần Gemini Teacher CoT; False = 2,724 mẫu hợp nhất toàn kho sạch
USE_PURE_TEACHER_DISTILL = True
dataset_file = "dataset_teacher_distilled.json" if USE_PURE_TEACHER_DISTILL else "dataset_clean_verified.json"
dataset_url = f"https://raw.githubusercontent.com/LamGianggg/TKLLS/main/training/{dataset_file}"

print(f">>> Đang tải {dataset_file} từ GitHub...")
urllib.request.urlretrieve(dataset_url, dataset_file)

with open(dataset_file, "r", encoding="utf-8") as f:
    training_data = json.load(f)

print(f">>> Nạp thành công {len(training_data)} mẫu Verilog sạch đã qua ModelSim Truth Oracle!")

dataset = Dataset.from_list(training_data)
dataset = dataset.map(formatting_prompts_func, batched = True, remove_columns=dataset.column_names)
print(">>> Đã format xong dữ liệu huấn luyện! Số lượng mẫu:", len(dataset))

## Bước 6: Khởi tạo `SFTTrainer` và Huấn luyện LoRA

In [ ]:
import inspect
from trl import SFTTrainer
from unsloth import is_bfloat16_supported

try:
    from trl import SFTConfig
    ConfigCls = SFTConfig
except ImportError:
    from transformers import TrainingArguments as ConfigCls

cfg_kwargs = dict(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 10,
    # max_steps = 20, # Bỏ comment nếu chỉ muốn test nhanh 1-2 phút
    num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 10,
    save_strategy = "no",
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = "outputs",
    report_to = "none",
)

cfg_sig = inspect.signature(ConfigCls.__init__).parameters
if "dataset_text_field" in cfg_sig:
    cfg_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in cfg_sig:
    cfg_kwargs["max_seq_length"] = max_seq_length
if "max_length" in cfg_sig:
    cfg_kwargs["max_length"] = max_seq_length
if "dataset_num_proc" in cfg_sig:
    cfg_kwargs["dataset_num_proc"] = 2
if "packing" in cfg_sig:
    cfg_kwargs["packing"] = False

trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
trainer_kwargs = dict(
    model = model,
    train_dataset = dataset,
    args = ConfigCls(**cfg_kwargs),
)
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_sig:
    trainer_kwargs["tokenizer"] = tokenizer
if "dataset_text_field" in trainer_sig and "dataset_text_field" not in cfg_kwargs:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in trainer_sig and "max_seq_length" not in cfg_kwargs:
    trainer_kwargs["max_seq_length"] = max_seq_length
if "dataset_num_proc" in trainer_sig and "dataset_num_proc" not in cfg_kwargs:
    trainer_kwargs["dataset_num_proc"] = 2
if "packing" in trainer_sig and "packing" not in cfg_kwargs:
    trainer_kwargs["packing"] = False

trainer = SFTTrainer(**trainer_kwargs)

print(">>> Bắt đầu huấn luyện SFT LoRA...")
trainer_stats = trainer.train()
print(">>> HUẤN LUYỆN SFT HOÀN TẤT!")

## Bước 6.5: Vẽ biểu đồ SFT Loss (tùy chọn lưu báo cáo)

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
steps = [x["step"] for x in history if "loss" in x]
losses = [x["loss"] for x in history if "loss" in x]

if steps and losses:
    plt.figure(figsize=(10, 4.5))
    plt.plot(steps, losses, color="#16a34a", linewidth=2, label="SFT Training Loss")
    plt.title("SFT Training Loss Convergence (Qwen2.5-Coder-3B)", fontsize=13, fontweight="bold")
    plt.xlabel("Training Steps")
    plt.ylabel("Loss")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.savefig("sft_loss_convergence.png", dpi=200)
    plt.show()

## Bước 7: Kiểm thử sinh mã Verilog sau SFT (Inference Test)

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Hãy thiết kế bộ đếm 8-bit có reset đồng bộ tích cực cao và ngõ ra cờ tràn tc khi count = 255.\n\n[Đặc tả thiết kế]:\nInputs: clk, rst. Outputs: [7:0] count, tc."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 768, use_cache = True)

## Bước 8: Lưu trọng số SFT LoRA (Dùng nối tiếp sang Notebook DPO hoặc tải về)

In [ ]:
# 1. Lưu vào spec2rtl_sft_lora để notebook DPO tự động nhận diện nếu chạy cùng phiên Colab
sft_dir = "spec2rtl_sft_lora"
model.save_pretrained(sft_dir)
tokenizer.save_pretrained(sft_dir)
print(f">>> Đã lưu SFT LoRA adapter vào thư mục: {sft_dir}")

# 2. Lưu bản dự phòng đúng tên thư mục local trong repo
local_dir = f"qwen2.5_{MODEL_SIZE.lower()}_verilog_lora"
model.save_pretrained(local_dir)
tokenizer.save_pretrained(local_dir)

!zip -qr spec2rtl_sft_lora.zip spec2rtl_sft_lora
!zip -qr {local_dir}.zip {local_dir}
print(f">>> Đã tạo {local_dir}.zip và spec2rtl_sft_lora.zip!")
print(">>> BƯỚC TIẾP THEO: Giữ nguyên phiên Colab này và mở notebook train_spec2rtl_qwen3b_dpo_colab.ipynb để train nối tiếp DPO!")